# Duka Electronics Sales Exploration

## Executive Summary
- The dataset contains **1,000 orders** across **5 branches**, providing a six-month view of sales activity.
- The highest-revenue branch is **Nairobi**, generating **KES 26,106,100** from **376 orders**.
- **Laptop** is the leading product by revenue at **KES 41,535,000**, while **Tablet** leads unit sales with **691 units**.
- Weekend trading accounts for **277 orders (27.7%)**, compared with **723 weekday orders**.
- The best revenue month is **2026-04 (KES 13,518,300)**, compared with **2026-06 (KES 10,903,400)** in the weakest month.

*These figures are calculated from the fixed-seed dataset generated by `generate_data.py`.*

## 1. First-five-minutes inspection

Before analysis, load the raw extract and inspect its dimensions, structure, missingness, distributions, categories and date coverage. This establishes whether the data is suitable for the requested calculations.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Shape:", sales.shape)
display(sales.head())
display(sales.tail())
sales.info()
display(sales.describe(include="all"))
display(sales.isna().sum().rename("missing_values").to_frame())
display(sales.nunique().rename("distinct_values").to_frame())
print("Date range:", sales["order_date"].min(), "to", sales["order_date"].max())
print("Branch proportions:")
display(sales["branch"].value_counts(normalize=True).mul(100).round(1))
print("Channel proportions:")
display(sales["channel"].value_counts(normalize=True).mul(100).round(1))

Shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


,order_id,order_date,branch,product,quantity,channel,unit_price
count,1000.000000,1000,1000,1000,1000.000000,1000,1000.000000
unique,NaN,NaN,5,5,NaN,3,NaN
top,NaN,NaN,Nairobi,Laptop,NaN,In-store,NaN
freq,NaN,NaN,376,220,NaN,579,NaN
mean,1500.500000,2026-03-30 08:44:09.600000,NaN,NaN,3.012000,NaN,23918.300000
min,1001.000000,2026-01-01 00:00:00,NaN,NaN,1.000000,NaN,1200.000000
25%,1250.750000,2026-02-14 00:00:00,NaN,NaN,2.000000,NaN,2500.000000
50%,1500.500000,2026-03-29 00:00:00,NaN,NaN,3.000000,NaN,15000.000000
75%,1750.250000,2026-05-14 00:00:00,NaN,NaN,4.000000,NaN,28000.000000
max,2000.000000,2026-06-29 00:00:00,NaN,NaN,5.000000,NaN,65000.000000


,missing_values
order_id,0
order_date,0
branch,0
product,0
quantity,0
channel,0
unit_price,0


,distinct_values
order_id,1000
order_date,179
branch,5
product,5
quantity,5
channel,3
unit_price,5


Date range: 2026-01-01 00:00:00 to 2026-06-29 00:00:00
Branch proportions:


branch
Nairobi    37.6
Mombasa    18.6
Nakuru     16.2
Kisumu     15.9
Eldoret    11.7
Name: proportion, dtype: float64

Channel proportions:


channel
In-store       57.9
Online         32.0
Phone order    10.1
Name: proportion, dtype: float64

### Inspection observations

- **Shape:** 1,000 rows and 7 columns; each row represents an order.
- **Types:** `order_date` is parsed as datetime; identifiers/categories are categorical or integer-like, and `unit_price` is numeric.
- **Missingness:** the generated Duka extract has no missing values in its seven source columns.
- **Date coverage:** orders span 1 January 2026 to 29 June 2026 (confirm from the displayed output).
- **Categories:** five branches, five products and three sales channels. Branch and channel proportions show the generated business mix; Nairobi and In-store are expected to be the largest categories because of the generator probabilities.

## 2. Derived columns

Revenue is quantity multiplied by unit price. Month, weekday and weekend status are derived from the order date. The Premium band is defined using unit price at or above KES 28,000.

In [2]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.to_period("M").astype(str)
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")
display(sales.head())

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,2026-06,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,2026-02,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,2026-01,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,2026-04,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,2026-03,Saturday,True,Value


## 3. Branch ranking by revenue and orders

Revenue reflects both order volume and the value/quantity of items sold. The order ranking can differ because branches may have different product mixes and quantities per order.

In [3]:
branch_summary = (sales.groupby("branch")
                  .agg(total_revenue=("revenue", "sum"),
                       orders=("order_id", "count"))
                  .sort_values("total_revenue", ascending=False))
order_ranking = sales.groupby("branch").size().sort_values(ascending=False)
display(branch_summary)
display(order_ranking.rename("orders").to_frame())
print("Same ranking?", list(branch_summary.index) == list(order_ranking.index))

,total_revenue,orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117


,orders
branch,
Nairobi,376
Mombasa,186
Nakuru,162
Kisumu,159
Eldoret,117


Same ranking? True


### Interpretation

Compare the two displayed rankings. A branch can process more orders but earn less revenue if its baskets contain lower-priced products or smaller quantities. Conversely, a branch with fewer orders can rank higher in revenue when its sales mix contains more laptops/tablets or larger quantities.

## 4. Top products by revenue and units sold

Revenue ranking rewards expensive products and quantity sold; unit ranking counts each item equally.

In [4]:
product_summary = (sales.groupby("product")
                   .agg(total_revenue=("revenue", "sum"),
                        units_sold=("quantity", "sum"))
                   .sort_values("total_revenue", ascending=False))
display(product_summary.head(3).rename_axis("Top products by revenue"))
display(product_summary.sort_values("units_sold", ascending=False).head(3)
        .rename_axis("Top products by units sold"))

,total_revenue,units_sold
Top products by revenue,,
Laptop,41535000,639
Tablet,19348000,691
Phone,8880000,592


,total_revenue,units_sold
Top products by units sold,,
Tablet,19348000,691
Laptop,41535000,639
Phone,8880000,592


### Interpretation

The two lists need not match. A high-priced product such as a laptop can dominate revenue even if it sells fewer units, while low-priced accessories can lead unit volume without generating the highest revenue.

## 5. Weekend versus weekday performance

In [5]:
weekend_summary = (sales.groupby("is_weekend")
                   .agg(orders=("order_id", "count"),
                        total_revenue=("revenue", "sum"),
                        average_order_revenue=("revenue", "mean")))
weekend_summary.index = weekend_summary.index.map({False: "Weekday", True: "Weekend"})
display(weekend_summary)

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


### Interpretation

Compare order counts, total revenue and average revenue per order. Total revenue is affected by both the number of orders and average revenue per order; a smaller number of weekend orders can still produce substantial revenue if the average basket is larger.

## 6. Best and worst revenue month

In [6]:
monthly_revenue = sales.groupby("month")["revenue"].sum().sort_values(ascending=False)
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()
best_value = monthly_revenue.max()
worst_value = monthly_revenue.min()
percentage_difference = (best_value - worst_value) / worst_value * 100
display(monthly_revenue.sort_index().rename("revenue").to_frame())
print(f"Best month: {best_month} (KES {best_value:,.0f})")
print(f"Worst month: {worst_month} (KES {worst_value:,.0f})")
print(f"Percentage difference relative to the worst month: {percentage_difference:.2f}%")

,revenue
month,
2026-01,11786600
2026-02,11772000
2026-03,12396800
2026-04,13518300
2026-05,11415400
2026-06,10903400


Best month: 2026-04 (KES 13,518,300)
Worst month: 2026-06 (KES 10,903,400)
Percentage difference relative to the worst month: 23.98%


### Interpretation

The percentage difference is calculated as `(best-month revenue − worst-month revenue) / worst-month revenue × 100`. This expresses how much higher the best month was relative to the worst month.

## 7. Premium online orders from Nairobi and Mombasa

In [7]:
premium_online = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()
print("Number of qualifying orders:", len(premium_online))
print("Total revenue (KES):", premium_online["revenue"].sum())
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
display(premium_online.head())

Number of qualifying orders: 83
Total revenue (KES): 10218000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,2026-01,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,2026-06,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,2026-04,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,2026-03,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,2026-02,Tuesday,False,Premium


## 8. Spreadsheet reconciliation

The workbook `excel/duka_branch_summary.xlsx` contains a branch summary with `SUMIFS`, a region lookup with `XLOOKUP`, and a branch-by-channel revenue summary. The branch and channel totals below provide the pandas benchmark. Open the workbook in Excel to calculate formulas. For a native PivotTable, select the Sales Data table, choose **Insert → PivotTable**, place `branch` in Rows, `channel` in Columns and `revenue` in Values (Sum).

In [8]:
pandas_branch_totals = sales.groupby("branch")["revenue"].sum().sort_values(ascending=False)
pandas_channel_totals = sales.groupby("channel")["revenue"].sum()
pandas_grand_total = sales["revenue"].sum()
display(pandas_branch_totals.to_frame("revenue"))
print(f"Pandas grand total: KES {pandas_grand_total:,.0f}")
print("Expected Excel Branch Summary grand total: KES {:,.0f}".format(pandas_branch_totals.sum()))
print("Reconciliation benchmark matches:", pandas_branch_totals.sum() == pandas_grand_total)
print("Pandas channel revenue totals:")
display(pandas_channel_totals.to_frame("revenue"))

,revenue
branch,
Nairobi,26106100
Mombasa,14003700
Nakuru,11970400
Kisumu,10868100
Eldoret,8844200


Pandas grand total: KES 71,792,500
Expected Excel Branch Summary grand total: KES 71,792,500
Reconciliation benchmark matches: True
Pandas channel revenue totals:


,revenue
channel,
In-store,40969200
Online,22786100
Phone order,8037200


### Reconciliation note

The Excel workbook formulas use the same 1,000 source rows and the same revenue definition (`quantity × unit_price`). After opening the workbook in Excel, confirm the branch grand total equals the pandas overall revenue shown above. The branch-by-channel sheet is a formula-driven pivot-style summary; use the instructions above to insert a native PivotTable if the assessor requires the Excel PivotTable object.